# 🧪 Test Notebook: Code Generator với Schema Hỗ Trợ
Kiểm thử khả năng sinh mã của `code_generator_node` khi sử dụng thông tin schema:
1. Chọn đúng cột giá trị khi bảng có nhiều cột nhờ so sánh mô tả cột (`useful_columns`)
2. Trích xuất trực tiếp `total_value` khi truy vấn theo tên danh mục (`sub_sections`)
3. Tự động tính tổng các hàng trong `range` khi `total_value` không có sẵn

*Toàn bộ kết quả log được ghi tự động ra file `test_code_generator_log.txt`.*

In [ ]:
import os
import sys
import io
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ==============================================================================
# Dual Output Logger: Ghi toàn bộ kết quả in ra cả Console và file .txt
# ==============================================================================
class DualOutputLogger:
    """Redirects sys.stdout so that all outputs are printed to console/notebook AND saved into a .txt log file."""
    def __init__(self, log_filepath: str = "pipeline_execution.txt", stream=None):
        if stream is not None:
            self.terminal = stream
        elif isinstance(sys.stdout, DualOutputLogger):
            self.terminal = sys.stdout.terminal
        else:
            self.terminal = sys.stdout if sys.stdout is not None else sys.__stdout__
        self.log_filepath = Path(log_filepath)
        self.log_filepath.parent.mkdir(parents=True, exist_ok=True)
        self.log_file = open(self.log_filepath, "a", encoding="utf-8", errors="replace")

    def write(self, message):
        try:
            if hasattr(self.terminal, "write"):
                self.terminal.write(message)
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.write(message)
                self.log_file.flush()
        except Exception:
            pass

    def flush(self):
        try:
            if hasattr(self.terminal, "flush"):
                self.terminal.flush()
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.flush()
        except Exception:
            pass

    def isatty(self) -> bool:
        if hasattr(self.terminal, "isatty"):
            try:
                return self.terminal.isatty()
            except Exception:
                return False
        return False

    def fileno(self):
        if hasattr(self.terminal, "fileno"):
            return self.terminal.fileno()
        raise io.UnsupportedOperation("fileno not supported")

    def readable(self) -> bool:
        return False

    def writable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return False

    @property
    def encoding(self):
        return getattr(self.terminal, "encoding", "utf-8")

    @property
    def errors(self):
        return getattr(self.terminal, "errors", "replace")

    def close(self):
        if hasattr(self, "log_file") and not self.log_file.closed:
            self.log_file.close()

    def __getattr__(self, attr):
        return getattr(self.terminal, attr)

LOG_FILE = PROJECT_ROOT / "notebooks" / "test_code_generator_log.txt"
if not isinstance(sys.stdout, DualOutputLogger):
    sys.stdout = DualOutputLogger(str(LOG_FILE))
print(f"📝 Đã kích hoạt ghi log tự động ra file: {LOG_FILE.resolve()}")

from pipeline.src.config import config
from pipeline.src.nodes.code_generator import (
    _resolve_value_column,
    _build_files_context,
    code_generator_node
)
from pipeline.src.nodes.executor import executor_node
print("✅ Đã nạp thành công các hàm từ Code Generator & Executor!")

### 1. Kiểm thử `_resolve_value_column()` với Schema nhiều cột

In [ ]:
table_schema = ["Ma_Doanh_Nghiep", "CHỈ TIÊU", "Năm nay", "Năm trước"]
first_row_values = {}
schema_multi_col = {
    "useful_columns": [
        {"column_name": "Năm nay", "column_description": "Số liệu báo cáo tài chính năm hiện tại 2023"},
        {"column_name": "Năm trước", "column_description": "Số liệu so sánh của năm tài chính trước đó 2022"}
    ],
    "sub_sections": []
}

# Test 1a: Truy vấn tiêu chí 'năm trước'
parsed_query_prev = {"tieu_chi_phu": "năm trước", "noi_dung": "Doanh thu"}
col_resolved = _resolve_value_column(
    table_schema=table_schema,
    first_row_values=first_row_values,
    parsed_query=parsed_query_prev,
    column_mapping={"label_column": "CHỈ TIÊU", "value_column": "Năm nay"},
    label_col="CHỈ TIÊU",
    schema=schema_multi_col
)
print(f"Tiêu chí 'năm trước' -> Cột chọn được: '{col_resolved}'")
assert col_resolved == "Năm trước", f"Kỳ vọng 'Năm trước', nhận được '{col_resolved}'"

# Test 1b: Truy vấn tiêu chí mô tả '2022'
parsed_query_2022 = {"tieu_chi_phu": "2022", "noi_dung": "Doanh thu"}
col_resolved_2022 = _resolve_value_column(
    table_schema=table_schema,
    first_row_values=first_row_values,
    parsed_query=parsed_query_2022,
    column_mapping={"label_column": "CHỈ TIÊU", "value_column": "Năm nay"},
    label_col="CHỈ TIÊU",
    schema=schema_multi_col
)
print(f"Tiêu chí '2022' (khớp qua description) -> Cột chọn được: '{col_resolved_2022}'")
assert col_resolved_2022 == "Năm trước", f"Kỳ vọng 'Năm trước', nhận được '{col_resolved_2022}'"
print("✅ Test 1: _resolve_value_column with schema PASS!")

### 2. Kiểm thử Sinh mã & Thực thi Code Generator với Section Data

In [ ]:
test_csv = PROJECT_ROOT / "pipeline" / "data" / "test_financial_statement.csv"

state_input = {
    "user_query": "Tổng tài sản ngắn hạn năm 2023 của VNM là bao nhiêu?",
    "parsed_query": {
        "ten_cong_ty": "VNM",
        "so_nam": ["2023"],
        "noi_dung": "I. TÀI SẢN NGẮN HẠN",
        "muc_tieu": "trich_xuat",
        "tieu_chi_phu": "31/12/2023"
    },
    "discovered_tables": [
        {
            "csv_path": str(test_csv),
            "Ten_Bang": "Bảng Cân đối kế toán VNM 2023",
            "Nam_Tai_Chinh": "2023"
        }
    ],
    "table_schema": ["Ma_Doanh_Nghiep", "Nam_Tai_Chinh", "0", "1", "2"],
    "first_row_values": {"0": "TÀI SẢN", "1": "31/12/2023", "2": "01/01/2023"},
    "column_mapping": {"label_column": "0", "value_column": "1"},
    "schema": {
        "useful_columns": [
            {"column_name": "1", "column_description": "Giá trị ngày 31/12/2023"},
            {"column_name": "2", "column_description": "Giá trị ngày 01/01/2023"}
        ],
        "sub_sections": [
            {"section_name": "I. TÀI SẢN NGẮN HẠN", "range": [3, 5], "total_value": 35000000.0},
            {"section_name": "II. TÀI SẢN DÀI HẠN", "range": [8, 9], "total_value": 25000000.0}
        ]
    }
}

# Chạy Code Generator Node
gen_state = code_generator_node(state_input, config)
print(f"Generated Code:\n{gen_state.get('generated_code')}")

# Chạy Executor Node
exec_state = executor_node(gen_state, config)
print(f"\nExecution Result:\n{exec_state.get('execution_result')}")
assert exec_state.get("status") == "success", f"Thực thi thất bại: {exec_state.get('error_traceback')}"
assert exec_state["execution_result"]["data"] == 35000000.0, f"Kỳ vọng 35000000.0, nhận được {exec_state['execution_result']['data']}"
print(f"📝 Đã ghi toàn bộ log kiểm thử ra file: {LOG_FILE.resolve()}")
print("✅ Test 2: Code Generator & Executor with Section PASS!")